# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
import numpy as np
import pandas as pd

# load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# clean data
df.columns = df.columns.str.lower().str.replace(" ", "_")

print("Database shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

#Low claim amount
print("\n" + "=" * 60)
print("Task 1: Low claim amount + Yes response")
print("=" * 60)

df_task1 = df.loc[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
]

print(f"Total customers : {len(df)}")
print(f"Filtered customers : {len[df_task1]}")
print(f"\nSample data:")
df_task1.head()

#Average monthly premium
print("\n" + "=" * 60)
print("Task 2: Average monthly premium and CLV analysis")
print("=" * 60)

#filter for yes response
df_yes = df.loc[df["response"] == "Yes"]

#average monthly premium
avg_premium = df_yes.groupby(["policy type", "gender"])[
    ["monthly_premium_auto", "customer_lifetime_value"]
].mean().round(2)

print("\nAverage Monthly Premium and CLV by Policy Type and Gender:")
print(avg_premium)

avg_claims = df_yes.groupby(["policy type", "gender"])[
    "total claim amount"
].mean().round(2)

print("\nAverage Total Claim Amount by Policy Type and Gender:")
print(avg_claims)

print("""
Conclusions:
- Segments with HIGH CLV and LOW claim amounts are most profitable
- Segments with LOW premiums and HIGH claims are most at risk
- Compare the two tables to identify the most profitable segments
""")

#task 3: policies per state
print("\n" + "=" * 60)
print("Task 3: Number of customers per state (> 5000)")
print("=" * 60)

customers_per_state = df.groupby("state")["customer"].count()
states_over_500 = customers_per_state[customers_per_state > 500]

print("\nStates with more than 500 customers:")
print(states_over_500.sort_values(ascending=False))

#Task 4: CLV by education level and gender
print("\n" + "=" * 60)
print("Task 4: CLV by education level and gender")
print("=" * 60)

clv_by_education = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["max", "min", "median"]).round(2)
clv_by_education.columns = ["Max CLV", "Min CLV", "Median CLV"]
 print("\nCLV Statistics by Education and Gender:")
print(clv_by_education)

print("""
Conclusions:
- Higher education levels may correlate with higher CLV
- Gender differences in CLV can indicate target demographics
- Large gaps between max and min suggest high variability in some segments
""")

#Task 5: Policies sold by state and month
print("\n" + "=" * 60)
print("Task 5: Policies sold by state and month")
print("=" * 60)

df["month"] = pd.to_datetime(df["effective_to_date"]).dt.month_name()

#create pivot table
policies_by_state_month = df.pivot_table(
    index="state",
    columns="month",
    values="customer",
    aggfunc="count",
    fill_value=0
)

print("\nPolicies Sold by State and Month:")
print(policies_by_state_month)

#Task 6: top 3 states by policies sold
print("\n" + "=" * 60)
print("Task 6: Top 3 states by policies sold")
print("=" * 60)

#group by state and month
policies_by_state_month.columns = df.groupby(["state", "month"])["customer"].count().reset_index()
policies_by_state_month.columns = ["state", "month", "policies_sold"]

top3_states = (
 policies_by_state_month.groupby("state")["policies_sold"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

print(f"\nTop 3 States: {top3_states}")

#filter for top 3 states
df_top3 = policies_by_state_month[policies_by_state_month["state"].isin(top3_states)]

#pivot for better readability
df_top3_pivot = df_top3.pivot_table(
    index="state",
    columns="month",
    values="policies_sold",
    fill_value=0
)
print("\nPolicies Sold by Month for top 3 states:")
print(df_top3_pivot)

#Task 7: Marketing channel response rate
print("\n" + "=" * 60)
print("Task 7: Marketing channel response rate")
print("=" * 60)

#melt the data to unpivot marketing channels
df_melted = df.melt(
    id_vars=["customer", "response"],
    value_vars=["sales_channel"],
    var_name="channel_type",
    value_name="channel"
)

#calculate response rate by channel
response_rate = df.melted.groupby("channel").apply(
    lambda x: (x["response"] == "Yes").sum() / len(x) * 100
).round(2).reset_index()
response_rate.columns = ["channels", "response_rate_%"]
response_rate = response_rate_sort_values("response_rate_%", ascending=False)

print("\nCustomer Response Rate by Marketing Channel:")
print(response_rate)
print("""Conclusions:
- Channels with highest rsponse rates are most effective
- Focus marketing budget on top performing channels
- Low response rate channels may need strategy review
""")